# Cambodian legal OCR → article dataset

This notebook converts OCR `.txt` files into one row per legal article. It keeps the original article wording, creates a conservative normalized search field, tracks the chapter/section context, and joins document metadata and PDF paths.

**Important:** `article_text` is not aggressively cleaned. Use `search_text` for retrieval and `article_text` for display/citation.

## 1. Setup and paths

Change only these paths when moving the notebook. The default paths match this repository.

In [28]:
from pathlib import Path
import csv
import re
import unicodedata
import hashlib
import pandas as pd
from IPython.display import display

from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
TESTABLE_DIR = PROJECT_ROOT / 'data-pipeline' / 'testable'
# Strict input: only final OCR files under data/ocr/text are allowed.
FINAL_TEXT_DIR = DATA_DIR / 'ocr' / 'text'
ORIGINAL_SOURCE_CSV = DATA_DIR / 'checkpoints' / 'documents.csv'
OUTPUT_DIR = DATA_DIR / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('Final OCR files:', len(list(FINAL_TEXT_DIR.glob('*.txt'))))

Project: /Users/nak/CADT/projects/legal-rag
Final OCR files: 95


## 2. Load document metadata

The metadata file is read as strings so document IDs, URLs, Khmer titles, and empty values are preserved. Each final OCR filename (`{document_id}.txt`) is joined to this original source row by `document_id`; no metadata is guessed.

In [29]:
documents = pd.read_csv(ORIGINAL_SOURCE_CSV, dtype='string', keep_default_na=False, encoding='utf-8-sig')
documents['document_id'] = documents['document_id'].str.strip()
documents = documents.drop_duplicates('document_id', keep='last').set_index('document_id')
final_document_ids = {p.stem for p in FINAL_TEXT_DIR.glob('*.txt')}
missing_source_rows = final_document_ids - set(documents.index)
assert not missing_source_rows, f'Final OCR files missing original source rows: {sorted(missing_source_rows)}'

def metadata_for(document_id):
    row = documents.loc[document_id].to_dict() if document_id in documents.index else {}
    row['document_id'] = document_id
    row['original_source_metadata'] = str(ORIGINAL_SOURCE_CSV)
    return row

display(documents.reset_index().head(3))

,document_id,title,detail_url,pdf_url,local_pdf_path,status,retry_count,error,document_type,publication_date,publication_year,language,category,category_id,updated_at
0,1,ច្បាប់ស្តីពីចរាចរផ្លូវគោក​ (វិសោធនកម្ម),https://library.ncdd.gov.kh/detail/1,https://library.ncdd.gov.kh/detail/1/download,/opt/legal-rag/data/pdfs/1.pdf,downloaded,3,,Law,,2024,Khmer,NCDD Laws,7,2026-10-28T16:13:15.738615+00:00
1,1004,ច្បាប់ស្តីពីប្រកាសនីយបត្រតក្កកម្មវិញ្ញាបនបត្រម...,https://library.ncdd.gov.kh/detail/1004,https://lib.ncdd.gov.kh/storage/app/public/lib...,/opt/legal-rag/data/pdfs/1004.pdf,downloaded,6,,Law,,2024,Khmer,NCDD Laws,7,2026-10-28T16:13:33.870656+00:00
2,1006,ច្បាប់ស្តីពីសិទ្ធិអ្នកនិពន្ធ និងសិទ្ធិប្រហាក់ប...,https://library.ncdd.gov.kh/detail/1006,https://lib.ncdd.gov.kh/storage/app/public/lib...,/opt/legal-rag/data/pdfs/1006.pdf,downloaded,6,,Law,,2024,Khmer,NCDD Laws,7,2026-10-28T16:13:49.678443+00:00


## 3. Normalize OCR without changing the legal display text

OCR commonly contains zero-width characters, non-breaking spaces, repeated whitespace, and inconsistent punctuation. The functions below separate three concerns:

- `clean_display_text`: safe whitespace cleanup for readable output.
- `search_text`: additionally removes punctuation/symbols and folds whitespace.
- Khmer and Arabic digits are both accepted in headings.

In [30]:
KHMER_DIGITS = str.maketrans('០១២៣៤៥៦៧៨៩', '0123456789')
ZERO_WIDTH = '\u200b\u200c\u200d\ufeff'

def unicode_clean(value):
    value = unicodedata.normalize('NFKC', str(value or ''))
    value = value.translate(str.maketrans('', '', ZERO_WIDTH))
    value = value.replace('\u00a0', ' ')
    return value

def clean_display_text(value):
    value = unicode_clean(value)
    value = re.sub(r'[ \t]+', ' ', value)
    value = re.sub(r' *\n *', '\n', value)
    return value.strip()

def search_text(value):
    value = clean_display_text(value).lower()
    # Keep Unicode letters, combining marks (important for Khmer), numbers, and spaces.
    value = ''.join(ch if (unicodedata.category(ch)[0] in {'L', 'M', 'N'} or ch.isspace()) else ' ' for ch in value)
    return re.sub(r'\s+', ' ', value).strip()

def number_value(value):
    digits = unicode_clean(value).translate(KHMER_DIGITS)
    return int(digits) if digits.isdigit() else None

## 4. Detect chapters, sections, and articles

The article boundary is the key unit for legal search. The regex allows spaces and common OCR punctuation, for example `មាត្រា១`, `មាត្រា ១៨._ ថ្មី`, and `មាត្រា 5.-`. Text before the first article is retained as `preamble` rather than silently discarded.

In [31]:
NUM = r'[0-9០-៩]+'
CHAPTER_RE = re.compile(r'^\s*ជំពូក\s*ទី\s*(' + NUM + r')\s*(?:[._៖:-]+)?\s*(.*)$')
SECTION_RE = re.compile(r'^\s*ផ្នែក\s*ទី\s*(' + NUM + r')\s*(?:[._៖:-]+)?\s*(.*)$')
ARTICLE_TOKEN_RE = re.compile(r'\[START_ARTICLE\s*:\s*(' + NUM + r')\s*\]')
ARTICLE_MARKER_RE = re.compile(r'^\s*\[START_ARTICLE\s*:\s*(' + NUM + r')\s*\]\s*$')

def heading_match(pattern, line):
    match = pattern.match(unicode_clean(line))
    if not match:
        return None
    return number_value(match.group(1)), clean_display_text(match.group(2))

def remove_article_heading(lines):
    # START_ARTICLE marks the article; remove its first OCR heading line.
    lines = list(lines)
    while lines and not lines[0].strip():
        lines.pop(0)
    if lines and re.match(r'^[\"“”]?\s*មាត្រា', lines[0]):
        lines.pop(0)
    return lines

def source_text(document_id):
    # Strict by design: no legacy text or chunk-text fallback is permitted.
    path = FINAL_TEXT_DIR / f'{document_id}.txt'
    if not path.is_file():
        raise FileNotFoundError(f'Final OCR text is missing: {path}')
    return path.read_text(encoding='utf-8-sig'), str(path)

DOCUMENT_NUMBER_RE = re.compile(r'(?m)^\s*([^\s]+/[^\s]+/[^\s]+/[^\s]+)\s*$')

def extract_document_number(text):
    preamble = ARTICLE_TOKEN_RE.split(unicode_clean(text), maxsplit=1)[0]
    match = DOCUMENT_NUMBER_RE.search(preamble)
    return match.group(1) if match else None

In [32]:
def parse_articles(document_id, text, text_path):
    rows = []
    preamble = []
    current = None
    # OCR can place a marker inline with the previous article. Normalize
    # every marker onto its own line before extracting article spans.
    normalized_text = ARTICLE_TOKEN_RE.sub(
        lambda match: f'\n[START_ARTICLE:{match.group(1)}]\n',
        unicode_clean(text),
    )
    chapter_number, chapter_title = None, ''
    section_number, section_title = None, ''

    def save_current():
        if current is None:
            return
        body_lines = remove_article_heading(current.pop('_lines'))
        # Keep the legal wording and line breaks, but remove OCR spacing/control noise.
        raw = clean_display_text('\n'.join(body_lines))
        current['article_text'] = raw
        rows.append(current.copy())

    for raw_line in normalized_text.splitlines():
        line = raw_line.strip()
        chapter = heading_match(CHAPTER_RE, line)
        section = heading_match(SECTION_RE, line)
        marker = ARTICLE_MARKER_RE.match(line)
        # Only START_ARTICLE markers define article boundaries.
        if marker:
            save_current()
            article_number = number_value(marker.group(1))
            current = {
                'document_id': str(document_id), 'article_number': article_number,
                'article_label': f'មាត្រា {article_number}', 'article_heading': '',
                'chapter_number': chapter_number, 'chapter_title': chapter_title,
                'section_number': section_number, 'section_title': section_title,
                'source_text_path': text_path, '_lines': []
            }
        elif chapter:
            if current is not None:
                save_current()
                current = None
            chapter_number, chapter_title = chapter
            section_number, section_title = None, ''
        elif section:
            if current is not None:
                save_current()
                current = None
            section_number, section_title = section
        elif current is not None:
            current['_lines'].append(raw_line)
        elif line:
            preamble.append(raw_line)
    save_current()
    return rows, clean_display_text('\n'.join(preamble))

## 5. Build the article table

One output row represents one article. The PDF path is metadata for citation/opening the source; it is not embedded inside the text field.

In [33]:
def build_dataset(document_ids=None):
    if document_ids is None:
        document_ids = sorted(p.stem for p in FINAL_TEXT_DIR.glob('*.txt'))
    records = []
    for document_id in map(str, document_ids):
        text, text_path = source_text(document_id)
        article_rows, preamble = parse_articles(document_id, text, text_path)
        meta = metadata_for(document_id)
        document_number = extract_document_number(text)
        has_markers = bool(ARTICLE_TOKEN_RE.search(unicode_clean(text)))
        common = {
            'document_id': document_id, 'document_type': meta.get('document_type', ''),
            'document_number': document_number, 'title': meta.get('title', ''),
            'issued_date': meta.get('publication_date', '') or '',
            'pdf_url': meta.get('pdf_url', ''), 'local_pdf_path': meta.get('local_pdf_path', ''),
            'source_text_path': text_path,
            'original_source_metadata': meta.get('original_source_metadata', '')
        }
        review_reason = 'No START_ARTICLE marker found' if not has_markers else ''
        if preamble.strip():
            records.append({**common, 'record_type': 'preamble', 'chapter_number': None,
                'chapter_title': None, 'section_number': None, 'section_title': None,
                'article_number': None, 'article_text': preamble,
                'review_required': not has_markers, 'review_reason': review_reason})
        for row in article_rows:
            records.append({**common, 'record_type': 'article',
                'chapter_number': row.get('chapter_number'), 'chapter_title': row.get('chapter_title'),
                'section_number': row.get('section_number'), 'section_title': row.get('section_title'),
                'article_number': row.get('article_number'), 'article_text': row.get('article_text', ''),
                'review_required': False, 'review_reason': ''})
    columns = [
        'document_id', 'document_type', 'document_number', 'title', 'issued_date', 'sequence',
        'record_type', 'chapter_number', 'chapter_title', 'section_number', 'section_title',
        'article_number', 'article_text', 'pdf_url', 'local_pdf_path', 'source_text_path',
        'review_required', 'review_reason'
    ]
    result = pd.DataFrame(records)
    result = result[result['article_text'].fillna('').str.strip().ne('')].reset_index(drop=True)
    result['sequence'] = result.groupby('document_id').cumcount() + 1
    result = result[columns]
    for column in ['chapter_number', 'section_number', 'article_number', 'sequence']:
        result[column] = pd.to_numeric(result[column], errors='coerce').astype('Int64')
    return result

articles = build_dataset()
print(f'{len(articles):,} articles from {articles.document_id.nunique():,} documents')
display(articles[['document_id', 'chapter_number', 'chapter_title', 'section_number', 'section_title', 'article_number', 'article_text']].head(5))

2,635 articles from 90 documents


,document_id,chapter_number,chapter_title,section_number,section_title,article_number,article_text
0,1,<NA>,,<NA>,,1,មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ...
1,1,<NA>,,<NA>,,8,គ្រប់អ្នកបើកបរយានយន្តគ្រប់ប្រភេទ ត្រូវអនុវត្តន...
2,1,<NA>,,<NA>,,40,ប័ណ្ណបើកបរជាតិសម្រាប់បើកបរយានយន្តក្នុងព្រះរាជា...
3,1,<NA>,,<NA>,,41,លក្ខខណ្ឌផ្សេងៗទាក់ទងនឹងប័ណ្ណបើកបរ ត្រូវកំណត់ដូ...
4,1,<NA>,,<NA>,,47,គ្រប់យានយន្តនៅពេលដាក់ឱ្យធ្វើចរាចរណ៍នៅលើផ្លូវថ្...


## 6. Quality checks before export

These checks catch OCR/parser failures early. Duplicate article numbers are allowed because they can occur in different documents; duplicates within one document deserve review.

In [34]:
assert not articles.empty, 'No articles were parsed'
assert articles['article_text'].str.len().gt(0).all(), 'Empty raw article text found'

duplicate_articles = (articles.groupby(['document_id', 'article_number']).size()
                      .reset_index(name='count').query('count > 1'))
missing_pdf = articles[articles['pdf_url'].eq('')][['document_id', 'title']]
print('Duplicate article numbers:', len(duplicate_articles))
print('Documents with missing PDF path:', missing_pdf['document_id'].nunique())
display(articles.groupby('document_id').size().rename('article_count').describe())
display(duplicate_articles.head(10))

Duplicate article numbers: 57
Documents with missing PDF path: 0


count     90.000000
mean      29.277778
std       40.327877
min        1.000000
25%        2.250000
50%       12.500000
75%       40.500000
max      248.000000
Name: article_count, dtype: float64

,document_id,article_number,count
81,1004,71,2
82,1004,72,2
83,1004,73,2
84,1004,74,2
85,1004,75,2
86,1004,76,2
87,1004,77,2
88,1004,78,2
89,1004,79,2
90,1004,80,2


## 7. Export CSV files

`legal_articles.csv` is the only exported dataset. Royal decrees, approval text, and other preamble text are preserved as `record_type = preamble`; legal articles are stored as `record_type = article`. CSV is written with UTF-8 BOM for easy opening in Excel while preserving Khmer.

In [35]:
articles_path = OUTPUT_DIR / 'legal_articles.csv'
articles.to_csv(articles_path, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_MINIMAL)
print(articles_path)

/Users/nak/CADT/projects/legal-rag/data/processed/legal_articles.csv


## 8. Small retrieval smoke test

Use rows with `record_type = article` for retrieval. Keep preamble records for audit and document reconstruction. Later, article rows can be loaded into SQLite/FTS5 or used as the source for embeddings.

In [36]:
query = 'សារព័ត៌មាន'
mask = articles['article_text'].str.contains(query, regex=False, na=False)
display(articles.loc[mask, ['document_id', 'chapter_title', 'article_number', 'article_text', 'pdf_url']].head(10))

,document_id,chapter_title,article_number,article_text,pdf_url
172,1006,,25,អ្នកនិពន្ធមិនអាចធ្វើការហាមឃាត់បានចំពោះ ៖\nក-កា...,https://lib.ncdd.gov.kh/storage/app/public/lib...
176,1006,,29,ដោយមិនគិតពីបទប្បញ្ញត្តិនៅក្នុងមាត្រា ២១ ច្បាប់...,https://lib.ncdd.gov.kh/storage/app/public/lib...
598,1053,,74,គ្រប់ប្រព័ន្ធផ្សព្វផ្សាយព័ត៌មាន រួមមានសារព័ត៌ម...,https://lib.ncdd.gov.kh/storage/app/public/lib...
708,1055,,39,គ្រប់ករណីដូចមានចែងក្នុងមាត្រា ៣៦ និងមាត្រា ៣៧ ...,https://lib.ncdd.gov.kh/storage/app/public/lib...
714,1061,,4,បញ្ញើប្រែសណីយ៍ត្រូវបានបែងចែកជាពីរប្រភេទគឺ បញ្ញ...,https://library.ncdd.gov.kh/detail/1061/download
1030,1111,,40,ត្រូវបនអនុញ្ញាតឱ្យចូលក្នុងការិយាល័យចុះឈ្មោះអ្ន...,https://lib.ncdd.gov.kh/storage/app/public/lib...
1419,11169,,2,[ ការកម្រិតកិច្ចការពារចំពោះស្នាដៃមួយចំនួន : ១....,https://library.ncdd.gov.kh/detail/11169/download
1429,11169,,10,[ការប្រើប្រាស់ស្នាដៃដោយសេរីមួយចំនួន ៖ ១. អក្កដ...,https://library.ncdd.gov.kh/detail/11169/download
1430,11169,,10,[ការប្រើប្រាស់ស្នាដៃដោយសេរីមួយចំនួនទៀត ៖ ១. អត...,https://library.ncdd.gov.kh/detail/11169/download
1695,114,,15,ច្បាប់នេះ ត្រូវបានប្រកាសជាការប្រញាប់ ៖\n\nធ្វើ...,https://library.ncdd.gov.kh/detail/114/download
